# Final list-aware experiment — CatBoost MAE features

**Purpose:** test three CatBoost MAE feature stacks and their fixed-weight replacement blends against the **existing full-strength single-fit fixed-origin winner**. The incumbent is `followup_cat_mae_xgb_lgb`, a 50% CatBoost MAE / 35% XGBoost spline/geo / 15% LightGBM spline/geo geometric blend (prior observed **3.8996% MdAPE** on 5,951 August–October 2025 homes).

This notebook is the **interactive interface** to `feature/listaware_final_mae_features.py`. It does not duplicate the training logic or change any weights, parameters, source predictions, or release models.

**Important:** August–October 2025 has been inspected before. This is an exploratory historical experiment, not a fresh out-of-sample confirmation. The original `outputs/listaware_fixed_origin_full/` full-strength baseline must exist with its verified manifest and predictions.

**How to use:** run cells in order, review settings and preflight, then set `RUN_TRAINING = True` in the configuration cell before running the training cell. Rerunning with exactly the same settings resumes from verified saved model predictions. Afterwards you can set it back to `False` to explore the results without training.


## 1 · Locate the repository

Works from `experiments/`, the repository root, or another Jupyter working directory if you set the optional environment variable `DATA_DASH_ROOT`. Paths are resolved relative to the **repository root**, not the notebook directory.


In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

start = Path(os.environ.get("DATA_DASH_ROOT") or Path.cwd()).expanduser().resolve()
root_candidates = [start, *start.parents]
ROOT = next((
    p for p in root_candidates
    if (p / "data.py").is_file()
    and (p / "feature" / "listaware_final_mae_features.py").is_file()
), None)
if ROOT is None:
    raise FileNotFoundError(
        "Cannot find the Data Dash checkout. Open the notebook within the repo "
        "or set DATA_DASH_ROOT to the repository's full path."
    )
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from feature.listaware_final_mae_features import (
    CHALLENGERS, REPLACEMENTS, BASE_WEIGHTS, MONTHS, WINNER,
    run as run_final_mae_features, _sha256,
)
from listaware_full_fixed_origin_confirmation import _verified_full_origin_manifest

print("Repository:", ROOT)
print("Validation months:", ", ".join(MONTHS))
print("Frozen incumbent:", WINNER)


## 2 · Experiment controls

The experiment is intentionally **full strength**: it trains exactly three new CatBoost MAE models once on sales completed before August 1, 2025, and evaluates all eligible August–October 2025 homes.

The original XGBoost and LightGBM models are **not** retrained. Their predictions, the prior CatBoost MAE predictions, and the original 50/35/15 winner are read from the existing baseline output folder.

The experiment's manifest fingerprints your source files and configuration. Changing settings later requires a different output directory.


In [ ]:
SOURCE_DIR = ROOT / "outputs" / "listaware_fixed_origin_full"
OUTPUT_DIR = ROOT / "outputs" / "listaware_final_mae_features"

THREADS = 4
SEED = 42
BOOTSTRAP_REPEATS = 2_000

RUN_TESTS = False    # Optional quick unit tests before training
RUN_TRAINING = False # Change to True to fit/resume the 3 full-strength CatBoost models

print("Original full-strength baseline:", SOURCE_DIR)
print("New experiment outputs:       ", OUTPUT_DIR)
print("Training enabled:             ", RUN_TRAINING)


## 3 · Predeclared candidates and blend replacements

Every challenger uses **CatBoost MAE** on the log sale-to-list-price ratio. For each challenger, the script substitutes it for the 50% MAE member of the frozen 50/35/15 geometric blend, with the **same original XGBoost and LightGBM predictions**.

- **Quota:** prefer comparable sales of the same house type.
- **Spatial stack:** add selected binary text indicators, space-time splines, and geographic context.
- **Combined:** spatial stack plus same-type comparable quota.

These are the only new model fits. No extra residual corrections, hyperparameter sweeps, or weight tuning.


In [ ]:
catalog = pd.DataFrame([
    {
        "candidate": name,
        "backend": spec.backend,
        "target": spec.target,
        "loss": spec.loss,
        "features": spec.feature_name,
    }
    for name, spec in CHALLENGERS.items()
])
display(catalog)
display(pd.DataFrame([
    {"replacement_blend": name, "replaces_50pct_mae_member": member}
    for name, member in REPLACEMENTS.items()
]))
print("Original frozen weights:", BASE_WEIGHTS)
assert BASE_WEIGHTS == {
    "cat_mae": 0.50, "xgb_spline_geo": 0.35, "lgb_spline_geo": 0.15
}


## 4 · Baseline preflight

The new experiment **requires** the existing full-strength single-fit fixed-origin manifest and per-home predictions. This check is read-only; it does not load the full housing dataset or fit any model.

If the preflight fails, check where you saved the original full-strength run and correct `SOURCE_DIR`. Do not substitute the expanding-origin or pilot predictions.


In [ ]:
baseline_file = SOURCE_DIR / "fixed_weight_followup" / "followup_predictions.csv"
manifest_file = SOURCE_DIR / "manifest.json"

if not manifest_file.is_file() or not baseline_file.is_file():
    print("Missing expected fixed-origin baseline file(s):")
    for item in (manifest_file, baseline_file):
        print(" ", "FOUND" if item.is_file() else "MISSING", item)
    print("Correct SOURCE_DIR before enabling training.")
    SOURCE_READY = False
else:
    baseline_manifest = _verified_full_origin_manifest(SOURCE_DIR)
    SOURCE_READY = True
    print("Verified original protocol:", baseline_manifest["mode"])
    print("Full strength:", baseline_manifest["full"])
    print("Fixed training cutoff (exclusive):", baseline_manifest["train_cutoff_exclusive"])
    print("Evaluation months:", ", ".join(baseline_manifest["months"]))
    print("Baseline saved predictions:", baseline_file)
    print("Dataset fingerprint:", baseline_manifest["data_sha256"][:16] + "…")


## 5 · Optional fast regression tests

Checks candidate feature-provider registration, exact blend weights, cohort alignment, fingerprint safety, and existing full-origin comparison code. No model training is required.


In [ ]:
if RUN_TESTS:
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "tests/test_listaware_final_mae_features.py",
            "tests/test_listaware_confirmation.py", "-q",
        ],
        check=True, cwd=ROOT
    )
else:
    print("Tests disabled. Set RUN_TESTS=True in the settings cell to run them.")


## 6 · Train or resume the experiment

**Set `RUN_TRAINING=True` in section 2 before executing this cell to start fitting.**

The runner validates the original manifest and historical listings fingerprint, then fits only missing candidates. It writes completed model/month prediction CSVs immediately. Running again with the same settings safely reuses those caches.

If `RUN_TRAINING=False`, the cell loads existing output tables, if present, without fitting anything.


In [ ]:
pooled = monthly = diagnostics = None

if RUN_TRAINING:
    if not SOURCE_READY:
        raise RuntimeError("Baseline preflight failed; do not train until it is fixed.")
    pooled, monthly, diagnostics = run_final_mae_features(
        source_dir=SOURCE_DIR,
        output_dir=OUTPUT_DIR,
        threads=THREADS,
        seed=SEED,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
    )
elif all((OUTPUT_DIR / name).is_file() for name in (
    "manifest.json",
    "pooled_vs_frozen_winner.csv",
    "monthly_vs_frozen_winner.csv",
    "error_correlations.csv",
)):
    # Avoid mixing previously saved results with a different baseline.
    existing_manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text())
    if SOURCE_READY:
        if (
            existing_manifest["source_manifest_sha256"] != _sha256(manifest_file)
            or existing_manifest["source_predictions_sha256"] != _sha256(baseline_file)
        ):
            raise ValueError("Saved results refer to a different baseline. Use a separate output folder.")
    pooled = pd.read_csv(OUTPUT_DIR / "pooled_vs_frozen_winner.csv")
    monthly = pd.read_csv(OUTPUT_DIR / "monthly_vs_frozen_winner.csv")
    diagnostics = pd.read_csv(OUTPUT_DIR / "error_correlations.csv")
    print("Loaded saved experiment reports; no model training performed.")
else:
    print("No saved results found. Set RUN_TRAINING=True in section 2, then rerun this cell.")


## 7 · Leaderboard versus the original winner

A **negative** `delta_winner_mdape_pp` means a candidate is better than the previous 3.8996% MdAPE winner on the same properties. Check MAPE and P90 APE too; a tiny median win with larger tail errors may not justify replacing a frozen model.

Paired confidence intervals here are exploratory because the historical cohort was previously inspected.


In [ ]:
if pooled is not None:
    columns = [
        "arm", "variant", "homes", "mdape", "mape", "p90_ape",
        "delta_winner_mdape_pp", "delta_winner_mape_pp",
        "delta_winner_mdape_ci_low", "delta_winner_mdape_ci_high",
        "homes_improved_vs_winner",
    ]
    display(pooled[[c for c in columns if c in pooled.columns]].round(4))
    selected = pooled.loc[
        pooled.variant.isin([WINNER, *REPLACEMENTS])
    ].sort_values("mdape")
    print("Incumbent versus the three replacement blends:")
    display(selected[[
        "variant", "mdape", "mape", "p90_ape",
        "delta_winner_mdape_pp", "delta_winner_mape_pp"
    ]].round(4))
else:
    print("Run or load results first (section 6).")


## 8 · Paired uncertainty and monthly stability

Inspect candidate-minus-incumbent MdAPE (percentage points) and the monthly breakdown. Confidence intervals crossing zero mean the apparent difference is not conclusive under the displayed paired bootstrap.


In [ ]:
import matplotlib.pyplot as plt

if pooled is not None and monthly is not None:
    subset = pooled.loc[pooled.variant.ne(WINNER)].sort_values(
        "delta_winner_mdape_pp"
    ).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(10, max(4, len(subset) * 0.45)))
    y = np.arange(len(subset))
    ax.hlines(
        y,
        subset.delta_winner_mdape_ci_low,
        subset.delta_winner_mdape_ci_high,
        linewidth=2,
    )
    ax.scatter(subset.delta_winner_mdape_pp, y, s=38)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_yticks(y, subset.variant)
    ax.invert_yaxis()
    ax.set_xlabel("MdAPE difference vs original winner (pp; negative = improvement)")
    ax.set_title("Paired bootstrap intervals — exploratory")
    plt.tight_layout()
    plt.show()

    monthly_table = monthly.pivot_table(
        index="variant", columns="month", values="delta_winner_mdape_pp"
    ).reindex(index=[WINNER, *REPLACEMENTS]).round(4)
    display(Markdown("**Monthly MdAPE change for the replacement blends**"))
    display(monthly_table)
else:
    print("Run or load results first (section 6).")


## 9 · Error complementarity diagnostics

Signed-log-residual correlation describes similarity to the existing winner's mistakes; it is **not** by itself a model admission criterion. Also check the fraction of homes with lower absolute percentage error.

Do not train a residual correction from these evaluation labels.


In [ ]:
if diagnostics is not None:
    display(diagnostics.sort_values(
        "homes_improved_fraction", ascending=False
    ).round(4))
else:
    print("Run or load results first (section 6).")


## 10 · Interpretation and next step

The candidate should be promoted **only** if it gives a meaningful improvement over the original blend, maintains reasonable monthly stability, and passes an appropriately separate chronological confirmation. Minor differences on this repeatedly inspected development cohort should not trigger another cycle of tuning.

**Output files:**

- `outputs/listaware_final_mae_features/pooled_vs_frozen_winner.csv`
- `outputs/listaware_final_mae_features/monthly_vs_frozen_winner.csv`
- `outputs/listaware_final_mae_features/error_correlations.csv`
- `outputs/listaware_final_mae_features/predictions.csv`

This notebook only controls the **list-aware** experiment. The existing AutoML ensemble factory remains **off-market only**.
